# Presentation Plots

One-graph-per-slide figures for the final presentation. All numbers are
pulled from cached result JSONs in `artifacts/results/` — this
notebook does **not** retrain anything.

Sections:
1. **MixStyle vs. baseline** — effect of adding MixStyle on top of ERM.
2. *(coming later)*
3. *(coming later)*


## Shared setup

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RESULTS_DIR = Path("artifacts/results")
FIG_DIR = Path("artifacts/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 200,
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

def load_result(name: str) -> dict:
    path = RESULTS_DIR / f"{name}.json"
    if not path.exists():
        raise FileNotFoundError(path)
    return json.loads(path.read_text(encoding="utf-8"))

def load_many(names):
    return [load_result(n) for n in names]


# Part 1 — Effect of adding MixStyle on the baseline

**Baseline.** ImageNet-pretrained ResNet-18 trained with ERM. We unfreeze
one residual stage + the classifier head and sweep which stage:
`Baseline-L1`, `Baseline-L2`, `Baseline-L3`. `Baseline-L2` is the
canonical reference — every MixStyle variant at `L2` shares this exact
training recipe except for the MixStyle module.

**MixStyle.** Perturbs the channel-wise feature statistics (mean/std)
inside the chosen layer with probability `p`, mixing weights drawn from
`Beta(a, a)`. We vary:
- **insertion layer** — `layer1`, `layer2`, `layer3`
- **mixing probability `p`** — 0.25, 0.5, 0.75
- **Beta concentration `a`** — 0.1, 0.3, 0.5

All 10 runs share the same split (site 18 held out as the unseen domain).


In [ ]:
# All three baselines (layer1 / layer2 / layer3 trainable) + every MixStyle variant.
BASELINE_NAMES = ["Baseline-L1", "Baseline-L2", "Baseline-L3"]
BASELINE_REF_NAME = "Baseline-L2"  # canonical baseline — reference line & delta anchor
MIXSTYLE_NAMES = [
    "MixStyle-L1-p0.5-a0.1",
    "MixStyle-L3-p0.5-a0.1",
    "MixStyle-L2-p0.25-a0.1",
    "MixStyle-L2-p0.5-a0.1",
    "MixStyle-L2-p0.75-a0.1",
    "MixStyle-L2-p0.5-a0.3",
    "MixStyle-L2-p0.5-a0.5",
]

def _short(name: str) -> str:
    return name.replace("MixStyle-", "MS-").replace("Baseline-", "Base-")

rows = []
for name in BASELINE_NAMES + MIXSTYLE_NAMES:
    r = load_result(name)
    rows.append({
        "name": name,
        "short": _short(name),
        "is_baseline": name in BASELINE_NAMES,
        "in_acc":  r["in_domain_test_acc"],
        "out_acc": r["out_domain_test_acc"],
        "best_val": r["best_val"],
    })
df = pd.DataFrame(rows)
baseline_ref_out = df.loc[df["name"] == BASELINE_REF_NAME, "out_acc"].iloc[0]
df["delta_out"] = df["out_acc"] - baseline_ref_out
df


## Figure 1 — Out-of-domain accuracy, sorted

Primary DG metric. Baseline is the gray bar; a dashed horizontal line
makes it easy to see which variants actually help on the unseen site.

In [ ]:
plot_df = df.sort_values("out_acc").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 4.5))
colors = ["#9e9e9e" if is_b else "#1f77b4" for is_b in plot_df["is_baseline"]]
bars = ax.bar(plot_df["short"], plot_df["out_acc"], color=colors, edgecolor="black", linewidth=0.6)

ax.axhline(baseline_ref_out, color="#9e9e9e", linestyle="--", linewidth=1.2,
           label=f"Baseline-L2 = {baseline_ref_out:.4f}")

for bar, v in zip(bars, plot_df["out_acc"]):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.0008, f"{v:.4f}",
            ha="center", va="bottom", fontsize=9)

# Legend proxy for the two color groups
from matplotlib.patches import Patch
legend_patches = [
    Patch(facecolor="#9e9e9e", edgecolor="black", label="Baseline (ERM)"),
    Patch(facecolor="#1f77b4", edgecolor="black", label="MixStyle variants"),
]
ax.legend(handles=legend_patches + [plt.Line2D([0],[0], color="#9e9e9e", linestyle="--",
          label=f"Baseline-L2 = {baseline_ref_out:.4f}")],
          loc="lower right", frameon=False, fontsize=9)

y_lo = plot_df["out_acc"].min() - 0.008
y_hi = plot_df["out_acc"].max() + 0.006
ax.set_ylim(y_lo, y_hi)
ax.set_ylabel("Out-of-domain accuracy (site 18)")
ax.set_title("Effect of MixStyle on the baseline — out-of-domain accuracy")
ax.tick_params(axis="x", rotation=30)
for lbl in ax.get_xticklabels():
    lbl.set_ha("right")

plt.tight_layout()
plt.savefig(FIG_DIR / "part1_mixstyle_out_acc.png", bbox_inches="tight")
plt.show()


## Figure 2 — In-domain vs out-of-domain, side-by-side

Shows the generalisation gap at a glance. Good MixStyle variants shrink
the gap (bars closer together) *and* raise out-of-domain accuracy.

In [ ]:
plot_df = df.sort_values("out_acc").reset_index(drop=True)
x = np.arange(len(plot_df))
w = 0.38

fig, ax = plt.subplots(figsize=(11, 4.5))
b1 = ax.bar(x - w/2, plot_df["in_acc"],  w, label="In-domain",      color="#4C72B0", edgecolor="black", linewidth=0.5)
b2 = ax.bar(x + w/2, plot_df["out_acc"], w, label="Out-of-domain",  color="#DD8452", edgecolor="black", linewidth=0.5)

# Mark baselines with a hatched pattern so they're visually distinct
for i, is_b in enumerate(plot_df["is_baseline"]):
    if is_b:
        b1[i].set_hatch("//"); b2[i].set_hatch("//")

for bars, vals in [(b1, plot_df["in_acc"]), (b2, plot_df["out_acc"])]:
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.0008, f"{v:.3f}",
                ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(plot_df["short"], rotation=30, ha="right")
ax.set_ylabel("Accuracy")
ax.set_ylim(plot_df[["in_acc","out_acc"]].values.min() - 0.01,
           plot_df[["in_acc","out_acc"]].values.max() + 0.01)
ax.set_title("Baseline vs MixStyle — in-domain vs out-of-domain accuracy\n(hatched bars = baseline ERM, solid = MixStyle)")
ax.legend(loc="lower right", frameon=False)

plt.tight_layout()
plt.savefig(FIG_DIR / "part1_mixstyle_in_vs_out.png", bbox_inches="tight")
plt.show()


NameError: name 'df' is not defined

## Figure 3 — Improvement over baseline (delta plot)

Cleanest "what did MixStyle buy us" slide. Positive bars = variant
beats baseline on the unseen site. The best variant is highlighted.

In [ ]:
# Deltas vs Baseline-L2 — include Baseline-L1/L3 for context so the reader sees
# the *baseline-to-baseline* spread next to the MixStyle gains.
plot_df = df[df["name"] != BASELINE_REF_NAME].sort_values("delta_out").reset_index(drop=True)
best_idx = plot_df["delta_out"].idxmax()

fig, ax = plt.subplots(figsize=(9, 4.5))
colors = []
for _, row in plot_df.iterrows():
    if row["is_baseline"]:
        colors.append("#9e9e9e")  # other baselines in gray
    elif row["delta_out"] > 0:
        colors.append("#2ca02c")  # MixStyle helps
    else:
        colors.append("#d62728")  # MixStyle hurts
colors[best_idx] = "#1a7a1a"  # highlight best

bars = ax.barh(plot_df["short"], plot_df["delta_out"], color=colors, edgecolor="black", linewidth=0.5)

for bar, v in zip(bars, plot_df["delta_out"]):
    ax.text(v + (0.0003 if v >= 0 else -0.0003), bar.get_y() + bar.get_height()/2,
            f"{v:+.4f}", ha="left" if v >= 0 else "right", va="center", fontsize=9)

ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel(f"Δ out-of-domain accuracy vs Baseline-L2 (= {baseline_ref_out:.4f})")
ax.set_title("MixStyle improvement over baseline  —  gray bars show baseline-L1/L3 spread for context")

plt.tight_layout()
plt.savefig(FIG_DIR / "part1_mixstyle_delta.png", bbox_inches="tight")
plt.show()


### Pick-one recommendation for the slide

Figure 3 is the cleanest single graph for "effect of adding MixStyle":
one quantity (Δ), one interpretation (positive = helps), with the
best variant obvious. Use Figure 1 if you also want absolute numbers on
the slide, and Figure 2 only if the generalisation-gap point matters.


# Shared plotting helpers

One plotting function for Parts 2–4 so every comparison graph looks
the same. Each call takes a list of result names, labels, and colors;
produces one bar chart (out-of-domain accuracy) with the baseline drawn
as a dashed reference line and the best-performing bar highlighted.


In [ ]:
BASELINE_NAME = "Baseline-L2"
MIXSTYLE_REF_NAME = "MixStyle-L2-p0.75-a0.1"  # the MixStyle config all regularisers were built on

GROUP_COLORS = {
    "baseline":   "#9e9e9e",
    "mixstyle":   "#1f77b4",
    "consistmix": "#2ca02c",
    "mmd":        "#d62728",
    "combined":   "#9467bd",
}

def baseline_out() -> float:
    return load_result(BASELINE_NAME)["out_domain_test_acc"]

def mixstyle_ref_out() -> float:
    return load_result(MIXSTYLE_REF_NAME)["out_domain_test_acc"]

def comparison_bar(
    entries: list[tuple[str, str, str]],
    title: str,
    save_name: str,
    ylabel: str = "Out-of-domain accuracy (site 18)",
    show_mixstyle_ref: bool = True,
    figsize: tuple[float, float] = (9.5, 4.5),
):
    """entries: list of (result_name, short_label, group_key)."""
    rows = []
    for name, label, group in entries:
        r = load_result(name)
        rows.append({"name": name, "label": label, "group": group,
                     "in_acc": r["in_domain_test_acc"],
                     "out_acc": r["out_domain_test_acc"]})
    d = pd.DataFrame(rows)

    best_idx = d["out_acc"].idxmax()
    colors = [GROUP_COLORS[g] for g in d["group"]]
    edgecolors = ["black"] * len(d)
    linewidths = [0.6] * len(d)
    # Highlight best bar with a thicker gold edge
    edgecolors[best_idx] = "#b8860b"
    linewidths[best_idx] = 2.0

    fig, ax = plt.subplots(figsize=figsize)
    bars = ax.bar(d["label"], d["out_acc"], color=colors,
                  edgecolor=edgecolors, linewidth=linewidths)

    base_out = baseline_out()
    ax.axhline(base_out, color=GROUP_COLORS["baseline"], linestyle="--",
               linewidth=1.2, label=f"Baseline = {base_out:.4f}")
    if show_mixstyle_ref:
        ms_out = mixstyle_ref_out()
        ax.axhline(ms_out, color=GROUP_COLORS["mixstyle"], linestyle=":",
                   linewidth=1.2, label=f"MixStyle (p=0.75,a=0.1) = {ms_out:.4f}")

    for bar, v in zip(bars, d["out_acc"]):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.0006,
                f"{v:.4f}", ha="center", va="bottom", fontsize=9)

    lo = d["out_acc"].min() - 0.006
    hi = d["out_acc"].max() + 0.006
    # make room for reference lines
    lo = min(lo, base_out - 0.003)
    hi = max(hi, base_out + 0.003)
    ax.set_ylim(lo, hi)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=25)
    for lbl in ax.get_xticklabels():
        lbl.set_ha("right")
    ax.legend(loc="lower right", frameon=False, fontsize=9)

    plt.tight_layout()
    plt.savefig(FIG_DIR / save_name, bbox_inches="tight")
    plt.show()
    return d


# Part 2 — ConsistMix: output-level consistency

**Idea.** MixStyle is *supposed* to be label-preserving — shuffling a
sample's channel-wise style statistics with another sample's should not
change the diagnosis. In practice, predictions flip. ConsistMix adds a
KL-divergence penalty that forces the model to agree with itself across
the two views:

`L = CE(logits_mixed, y) + λ · KL(softmax(logits_mixed) || softmax(logits_clean).detach())`

We treat the clean branch as a stop-gradient teacher. We sweep λ on
top of the best plain-MixStyle config (`L2, p=0.75, a=0.1`).


## Figure — Baseline vs MixStyle vs ConsistMix sweep

In [ ]:
consistmix_entries = [
    (BASELINE_NAME,                   "Baseline",         "baseline"),
    (MIXSTYLE_REF_NAME,               "MixStyle",         "mixstyle"),
    ("ConsistMix-L2-lam0.05",         "Consist λ=0.05", "consistmix"),
    ("ConsistMix-L2-lam0.1",          "Consist λ=0.1",  "consistmix"),
    ("ConsistMix-L2-lam0.5",          "Consist λ=0.5",  "consistmix"),
    ("ConsistMix-L2-lam1.0",          "Consist λ=1.0",  "consistmix"),
    ("ConsistMix-L2-lam2.0",          "Consist λ=2.0",  "consistmix"),
    ("ConsistMix-L2-lam5.0",          "Consist λ=5.0",  "consistmix"),
]

_ = comparison_bar(
    consistmix_entries,
    title="ConsistMix vs MixStyle vs Baseline — out-of-domain accuracy",
    save_name="part2_consistmix.png",
)


## Optional — λ sweep as a line plot

Useful if you want to show that the gain is not monotonic — too
much consistency starts hurting.

In [ ]:
lam_names = [
    ("ConsistMix-L2-lam0.01", 0.01),
    ("ConsistMix-L2-lam0.05", 0.05),
    ("ConsistMix-L2-lam0.1",  0.1),
    ("ConsistMix-L2-lam0.2",  0.2),
    ("ConsistMix-L2-lam0.5",  0.5),
    ("ConsistMix-L2-lam1.0",  1.0),
    ("ConsistMix-L2-lam2.0",  2.0),
    ("ConsistMix-L2-lam3.0",  3.0),
    ("ConsistMix-L2-lam5.0",  5.0),
    ("ConsistMix-L2-lam8.0",  8.0),
]
xs = [lam for _, lam in lam_names]
ys = [load_result(n)["out_domain_test_acc"] for n, _ in lam_names]

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.plot(xs, ys, marker="o", color=GROUP_COLORS["consistmix"], linewidth=2, label="ConsistMix")
ax.axhline(baseline_out(),      color=GROUP_COLORS["baseline"], linestyle="--", label=f"Baseline = {baseline_out():.4f}")
ax.axhline(mixstyle_ref_out(),  color=GROUP_COLORS["mixstyle"], linestyle=":",  label=f"MixStyle = {mixstyle_ref_out():.4f}")
ax.set_xscale("log")
ax.set_xlabel("λ (consistency weight)")
ax.set_ylabel("Out-of-domain accuracy")
ax.set_title("ConsistMix — effect of consistency weight")
ax.legend(loc="lower left", frameon=False, fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / "part2_consistmix_lambda_sweep.png", bbox_inches="tight")
plt.show()


# Part 3 — MMD: feature-level domain alignment

**Idea.** Rather than regularising at the output, align the *feature
distributions* across training sites. We compute a multi-kernel
Gaussian MMD between the pre-FC features of different sites inside
every batch and add it to the loss:

`L = CE + λₘₘd · MMD(features[site_i] || features[site_j])`

This is class-conditional: we only compare features from the same
label across sites. Applied *on top of* the same MixStyle base so the
model gets style perturbation and distribution alignment together
(`MixMMD`). We also include an MMD-only variant for contrast.


## Figure — Baseline vs MixStyle vs MMD variants

In [ ]:
mmd_entries = [
    (BASELINE_NAME,                 "Baseline",           "baseline"),
    (MIXSTYLE_REF_NAME,             "MixStyle",           "mixstyle"),
    ("MMD-only-L2-lam0.10-cc",      "MMD-only λ=0.10", "mmd"),
    ("MixMMD-L2-lam0.05",           "MixMMD λ=0.05",   "mmd"),
    ("MixMMD-L2-lam0.1",            "MixMMD λ=0.1",    "mmd"),
    ("MixMMD-L2-lam0.5",            "MixMMD λ=0.5",    "mmd"),
]

_ = comparison_bar(
    mmd_entries,
    title="MMD vs MixStyle vs Baseline — out-of-domain accuracy",
    save_name="part3_mmd.png",
)


### Takeaway
- **MMD-only** barely matches plain MixStyle — alignment without
  style perturbation leaves the backbone exposed to feature-statistic
  shifts at test time.
- **MixMMD** (MixStyle + MMD) is the overall best performer in the
  entire study (out-acc 0.9031).

# Part 4 — ConsistMix + MMD (both regularisers at once)

**Idea.** ConsistMix enforces prediction stability under style
perturbation; MMD aligns feature distributions across training sites.
They target different failure modes — do they compose?

`L = CE + λ_c · KL(consistency) + λₘₘd · MMD`

We sweep `(λ_c, λₘₘd)` on the same MixStyle base.


## Figure — Single regularisers vs their combination

In [ ]:
combined_entries = [
    (BASELINE_NAME,                   "Baseline",               "baseline"),
    (MIXSTYLE_REF_NAME,               "MixStyle",               "mixstyle"),
    ("ConsistMix-L2-lam1.0",          "ConsistMix (best)",      "consistmix"),
    ("MixMMD-L2-lam0.1",              "MixMMD (best)",          "mmd"),
    ("MixConsistMMD-L2-c0.25-m0.025", "Consist+MMD c0.25/m0.025", "combined"),
    ("MixConsistMMD-L2-c0.25-m0.05",  "Consist+MMD c0.25/m0.05",  "combined"),
    ("MixConsistMMD-L2-c0.5-m0.025",  "Consist+MMD c0.5/m0.025",  "combined"),
    ("MixConsistMMD-L2-c0.5-m0.05",   "Consist+MMD c0.5/m0.05",   "combined"),
]

_ = comparison_bar(
    combined_entries,
    title="ConsistMix + MMD vs singletons vs Baseline — out-of-domain accuracy",
    save_name="part4_consist_mmd.png",
    figsize=(10.5, 4.8),
)


### Takeaway
- Combining the two regularisers gives a **modest** boost over MixStyle
  alone (≈20.89 → 0.898 range) but does **not** beat the best
  single-regulariser variant (MixMMD λ=0.1, 0.9031).
- Interpretation: the two losses target overlapping failure modes more
  than complementary ones. Adding both eats capacity without much extra
  gain.
